# Used Car Price Prediction — End-to-End Regression Analysis
### CSE4114 — Introduction to Data Science

**Dataset:** `car_dataset.csv` — used-car listings from the Bangladeshi resale market
**Task type:** Supervised Learning → **Regression** (target = `price`)

---

**Notebook roadmap**

| # | Section |
|---|---------|
| 1 | Problem Statement and Project Objectives |
| 2 | Dataset Description and Source |
| 3 | Data Collection |
| 4 | Data Cleaning and Preprocessing |
| 5 | Exploratory Data Analysis (EDA) |
| 6 | Data Visualization |
| 7 | Feature Engineering |
| 8 | Feature Selection |
| 9 | Machine Learning Model Development |
| 10 | Model Evaluation |
| 11 | Results, Discussion, and Conclusion |

> **How to run:** `Runtime → Run all`. When the upload widget appears in Section 3, choose `car_dataset.csv` from your computer.

---
# 1. Problem Statement and Project Objectives

## 1.1 Problem Statement

Pricing a used car is hard. Unlike new cars, there is no manufacturer sticker price — the value of a
second-hand vehicle depends on a tangle of factors: how old it is, how far it has been driven, the size
of its engine, the brand's reputation for reliability, whether it runs on CNG or octane, and the body
style buyers currently want.

In Bangladesh this problem is sharper than in most markets. Steep import duties mean a reconditioned
Toyota can hold value for a decade, CNG conversion is common and materially changes running costs, and
most transactions happen through informal classified listings where the asking price is set by the
seller's intuition rather than by any reference data. The result is a market with **wide, inconsistent
price dispersion for near-identical vehicles**. Buyers overpay, sellers underprice, and neither side has
an objective benchmark.

**The question this project answers:**
> Given a used car's observable specifications — brand, model year, kilometers driven, engine capacity,
> transmission, body type and fuel system — can we predict its market price accurately enough to be a
> useful valuation aid?

Because the target variable (`price`) is **continuous**, this is a **regression** problem, not
classification or clustering.

## 1.2 Project Objectives

1. **Acquire and inspect** the used-car dataset, documenting its structure, size and provenance.
2. **Clean and preprocess** the data — resolve missing values, drop duplicate listings, and treat the
   extreme outliers that are characteristic of price data.
3. **Explore** the data statistically to understand distributions, relationships and the direction of
   each predictor's effect on price.
4. **Visualise** the key patterns so the findings are interpretable to a non-technical reader.
5. **Engineer features** that capture domain knowledge the raw columns do not — vehicle age, usage
   intensity, multi-fuel capability, brand tier.
6. **Select** the most informative subset of features using statistical and model-based methods.
7. **Build and compare** several regression models, from an interpretable linear baseline to
   non-linear ensemble methods.
8. **Evaluate** every model on a held-out test set using R², RMSE, MAE and MAPE, backed by
   cross-validation to guard against a lucky split.
9. **Conclude** with the best-performing model, the drivers of price it identifies, the honest
   limitations of the study, and directions for future work.

## 1.3 Success Criteria

| Criterion | Target |
|-----------|--------|
| Primary metric | Test **R² ≥ 0.80** on unseen listings |
| Error magnitude | **MAPE ≤ 20%** — within a fifth of the true asking price |
| Robustness | Cross-validated R² standard deviation kept small (no over-fitting to one split) |
| Interpretability | A ranked, explainable list of the factors that drive resale price |

---
# 2. Dataset Description and Source

## 2.1 Source

The dataset is a scrape of **used-car classified advertisements from the Bangladeshi online resale
market** (Bikroy / Carmudi-style listing portals), distributed as a flat CSV file named
`car_dataset.csv`. Each row is one advertisement posted by a seller.

- **Format:** CSV, comma-separated, UTF-8 with BOM
- **Rows:** ~1,209 listings
- **Columns:** 10
- **Target variable:** `price` (Bangladeshi Taka, BDT)
- **Licence/usage:** public classified listings, used here for academic coursework only

## 2.2 Column Dictionary

| # | Column | Type | Description |
|---|--------|------|-------------|
| 1 | `car_name` | Text | Full listing title, e.g. *"Toyota Fielder 2011"* — brand + model + a year |
| 2 | `brand` | Categorical | Manufacturer, e.g. Toyota, Nissan, Honda |
| 3 | `car_model` | Categorical | Model line, e.g. Fielder, Noah, Axio |
| 4 | `model_year` | Numeric (int) | Registration/model year of the vehicle |
| 5 | `transmission` | Categorical | `Automatic` or `Manual` |
| 6 | `body_type` | Categorical | Saloon, SUV / 4x4, MPV, Hatchback, Estate, Convertible, Coupé/Sports |
| 7 | `fuel_type` | Categorical (**multi-label**) | Comma-separated fuel systems, e.g. *"CNG, Octane"* |
| 8 | `engine_capacity` | Numeric (int) | Engine displacement in cubic centimetres (cc) |
| 9 | `kilometers_run` | Numeric (int) | Odometer reading in kilometres |
| 10 | `price` | Numeric (int) | **Target** — asking price in BDT |

## 2.3 Notable Characteristics (and why they matter)

- **`fuel_type` is multi-label, not single-label.** A value like `"Petrol, CNG, Octane"` means the car
  can run on all three. Naive one-hot encoding would produce 24 sparse, mostly-useless dummy columns.
  Section 7 handles this properly with **multi-hot encoding**.
- **`body_type` contains missing values** stored as the literal string `null`.
- **Heavy brand imbalance** — Toyota dominates the Bangladeshi reconditioned-import market and accounts
  for the large majority of listings.
- **`price` is strongly right-skewed** — most cars cluster around 1–2 million BDT with a thin luxury
  tail reaching 16 million.
- **`car_name` sometimes disagrees with `model_year`** (the title year and the field year differ). We
  treat `model_year` as authoritative and drop `car_name` as a redundant high-cardinality identifier.

---
# 3. Data Collection

The data was **collected by scraping public classified listings** rather than generated experimentally,
so "collection" here means reliably getting the CSV into the Colab runtime.

The cell below uses `google.colab.files.upload()` to open a file-picker widget. It is written
defensively so the same notebook also runs outside Colab (local Jupyter, VS Code) by falling back to a
local path.

> **Action required:** run the next cell and select `car_dataset.csv` from your machine.

### 3.1 Import libraries

All third-party libraries are imported once, up front, so every later cell can assume they exist.
Global plotting defaults and the random seed are fixed here too, which makes every run of this notebook
reproducible.

In [ ]:
# --- Core data handling ---
import pandas as pd
import numpy as np
import warnings, io, os, re

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Scikit-learn: preprocessing ---
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# --- Scikit-learn: models ---
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# --- Scikit-learn: feature selection & metrics ---
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             r2_score, mean_absolute_percentage_error)

# --- Global settings ---
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi']     = 110
plt.rcParams['axes.titlesize']  = 13
plt.rcParams['axes.titleweight'] = 'bold'
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
pd.options.display.float_format = '{:,.2f}'.format

print('All libraries imported successfully.')
print('pandas', pd.__version__, '| numpy', np.__version__)

### 3.2 Upload the dataset (Colab file-picker)

Running this cell opens a **Choose Files** button. Select `car_dataset.csv`.
If the notebook is not running in Colab, it silently falls back to looking for the file in the working
directory instead, so nothing breaks.

In [ ]:
def load_dataset():
    """Upload a CSV via the Colab widget, or fall back to a local file.

    Returns
    -------
    (pandas.DataFrame, str) : the loaded data and the filename it came from.
    """
    try:
        from google.colab import files          # only exists inside Colab
        print('Colab detected — please choose your CSV file.\n')
        uploaded = files.upload()               # opens the file-picker widget
        fname = next(iter(uploaded))
        data  = pd.read_csv(io.BytesIO(uploaded[fname]))
        print(f'\nUploaded and loaded: {fname}')
        return data, fname

    except ImportError:
        # Not running in Colab — look for the file locally.
        for candidate in ['car_dataset.csv', 'data/car_dataset.csv',
                          '/content/car_dataset.csv']:
            if os.path.exists(candidate):
                data = pd.read_csv(candidate)
                print(f'Not in Colab — loaded local file: {candidate}')
                return data, candidate
        raise FileNotFoundError(
            'car_dataset.csv not found. Place it beside this notebook and re-run.')

df_raw, source_file = load_dataset()
print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

### 3.3 First look at the raw data

Before touching anything we take an honest snapshot: the first rows, a random sample (the head of a
sorted file can be misleading), and the tail.

In [ ]:
print('=' * 70); print('FIRST 5 ROWS'); print('=' * 70)
display(df_raw.head())

print('=' * 70); print('RANDOM SAMPLE OF 5 ROWS'); print('=' * 70)
display(df_raw.sample(5, random_state=RANDOM_STATE))

print('=' * 70); print('LAST 5 ROWS'); print('=' * 70)
display(df_raw.tail())

### 3.4 Structure, data types and memory footprint

`.info()` confirms the dtype pandas inferred for each column and how many non-null entries it holds —
our first quantitative signal of where missing data lives.

In [ ]:
print('=' * 70); print('DATAFRAME INFO'); print('=' * 70)
df_raw.info()

print('\nColumn names :', list(df_raw.columns))
print('Memory usage :', f'{df_raw.memory_usage(deep=True).sum() / 1024:,.1f} KB')

numeric_cols     = df_raw.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df_raw.select_dtypes(exclude=np.number).columns.tolist()
print('\nNumeric columns     :', numeric_cols)
print('Categorical columns :', categorical_cols)

### 3.5 Summary statistics

Two separate summaries — one for numeric columns (spread, central tendency, range) and one for
categorical columns (cardinality, most frequent value). Read these with an eye for anything
*impossible*: a zero engine, a negative price, a model year in the future.

In [ ]:
print('=' * 70); print('NUMERIC SUMMARY'); print('=' * 70)
display(df_raw.describe().T)

print('=' * 70); print('CATEGORICAL SUMMARY'); print('=' * 70)
display(df_raw.describe(include='object').T)

---
# 4. Data Cleaning and Preprocessing

Raw listing data is written by thousands of individual sellers, so it is messy by construction. This
section works through three cleaning tasks in order:

1. **Missing values** — find them, understand *why* they are missing, impute or drop.
2. **Duplicates** — identical re-posted advertisements that would leak between train and test sets.
3. **Outliers** — genuinely extreme but plausible luxury cars, versus data-entry errors.

We work on a copy (`df`) so `df_raw` stays available for before/after comparison.

### 4.1 Create a working copy

Never mutate the original. Every cleaning step from here modifies `df`; `df_raw` remains the untouched
reference we compare against at the end of the section.

In [ ]:
df = df_raw.copy()
print(f'Working copy created. Starting shape: {df.shape}')

### 4.2 Quantify missing values

We count nulls per column as both an absolute number and a percentage. The percentage is what drives
the decision: a column missing 2% of values can be imputed safely; one missing 60% usually cannot.

Note that pandas already reads the literal string `null` in this file as `NaN`, so a plain
`.isnull()` count is accurate — but we also explicitly scan for sentinel strings like `'null'`, `'NA'`
and `'?'` that some scrapers leave behind, since those would hide inside object columns undetected.

In [ ]:
def missing_report(data):
    """Return a per-column missing-value table, worst offenders first."""
    total   = data.isnull().sum()
    percent = (total / len(data) * 100).round(2)
    report  = pd.DataFrame({'missing_count': total, 'missing_percent': percent})
    return report[report.missing_count > 0].sort_values('missing_count', ascending=False)

report = missing_report(df)
print('=' * 70); print('MISSING VALUE REPORT'); print('=' * 70)
if report.empty:
    print('No missing values detected by isnull().')
else:
    display(report)

# Hunt for disguised missing values stored as strings.
sentinels = ['null', 'NULL', 'None', 'NA', 'N/A', 'na', '?', '-', '']
print('\nDisguised missing values (sentinel strings):')
found = False
for col in df.select_dtypes(exclude=np.number).columns:
    hits = df[col].astype(str).str.strip().isin(sentinels).sum()
    if hits:
        print(f'  {col:<18}: {hits} sentinel value(s)')
        found = True
if not found:
    print('  None found — all missing values are proper NaN.')

print(f'\nTotal missing cells: {df.isnull().sum().sum()}')
print(f'Rows with at least one missing value: {df.isnull().any(axis=1).sum()}')

### 4.3 Visualise the missing-data pattern

A heatmap shows *where* the gaps sit. Scattered stripes suggest data missing at random (safe to
impute); solid horizontal bands would suggest whole rows failed to scrape (better to drop).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(df.isnull(), cbar=False, yticklabels=False,
            cmap='viridis', ax=axes[0])
axes[0].set_title('Missing-value map (yellow = missing)')
axes[0].set_xlabel('Column')

miss_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
sns.barplot(x=miss_pct.values, y=miss_pct.index, ax=axes[1], color='#4C72B0')
axes[1].set_title('Missing percentage by column')
axes[1].set_xlabel('% missing')
for i, v in enumerate(miss_pct.values):
    if v > 0:
        axes[1].text(v + 0.05, i, f'{v:.2f}%', va='center', fontsize=9)

plt.tight_layout(); plt.show()

### 4.4 Handle missing values

**Strategy and justification**

| Column | Missing | Treatment | Why |
|--------|---------|-----------|-----|
| `body_type` | ~1.5% | Impute with the label `'Unknown'` | Too few to lose the rows; "unknown body type" is itself informative — sellers who omit it may list differently |
| any numeric | if present | Impute with the **median** | The median is robust to the heavy right-skew in price and kilometres; the mean would be dragged by luxury outliers |
| any other categorical | if present | Impute with the **mode** | Preserves the existing category distribution |

Imputing rather than dropping keeps all 1,209 listings — valuable in a dataset this small.

In [ ]:
before_missing = df.isnull().sum().sum()

# --- Categorical: body_type gets an explicit 'Unknown' category ---
if 'body_type' in df.columns and df['body_type'].isnull().any():
    n = df['body_type'].isnull().sum()
    df['body_type'] = df['body_type'].fillna('Unknown')
    print(f"body_type      : {n} missing -> filled with 'Unknown'")

# --- Any remaining categorical columns: mode imputation ---
for col in df.select_dtypes(exclude=np.number).columns:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        n = df[col].isnull().sum()
        df[col] = df[col].fillna(mode_val)
        print(f'{col:<15}: {n} missing -> filled with mode "{mode_val}"')

# --- Numeric columns: median imputation ---
for col in df.select_dtypes(include=np.number).columns:
    if df[col].isnull().any():
        med = df[col].median()
        n = df[col].isnull().sum()
        df[col] = df[col].fillna(med)
        print(f'{col:<15}: {n} missing -> filled with median {med:,.0f}')

print(f'\nMissing cells before: {before_missing}')
print(f'Missing cells after : {df.isnull().sum().sum()}')
assert df.isnull().sum().sum() == 0, 'Missing values remain!'
print('All missing values resolved.')

### 4.5 Detect and remove duplicate listings

Duplicates matter more than they look. If the same advertisement appears twice and one copy lands in
the training set while the other lands in the test set, the model is effectively tested on data it has
memorised — inflating the score and hiding real error.

We check two definitions:
- **Exact duplicates** — every column identical.
- **Specification duplicates** — same car spec *and* same price, ignoring the free-text title.

In [ ]:
exact_dupes = df.duplicated().sum()
print(f'Exact duplicate rows (all columns): {exact_dupes}')

spec_cols = [c for c in ['brand', 'car_model', 'model_year', 'transmission',
                         'body_type', 'fuel_type', 'engine_capacity',
                         'kilometers_run', 'price'] if c in df.columns]
spec_dupes = df.duplicated(subset=spec_cols).sum()
print(f'Duplicate rows on specification + price: {spec_dupes}')

if exact_dupes:
    print('\nExample duplicate rows:')
    display(df[df.duplicated(keep=False)].sort_values(spec_cols[:3]).head(6))

rows_before = len(df)
df = df.drop_duplicates(subset=spec_cols, keep='first').reset_index(drop=True)
print(f'\nRows before: {rows_before:,}')
print(f'Rows after : {len(df):,}  (removed {rows_before - len(df):,})')

### 4.6 Sanity-check for impossible values

Before treating statistical outliers we remove *logically* invalid records — values that cannot exist
regardless of how the distribution looks. A car with a zero-cc engine or a negative price is a
data-entry error, not a rare vehicle.

In [ ]:
print('=' * 70); print('LOGICAL VALIDITY CHECKS'); print('=' * 70)

checks = {
    'price <= 0'                 : (df['price'] <= 0).sum(),
    'engine_capacity <= 0'       : (df['engine_capacity'] <= 0).sum(),
    'kilometers_run < 0'         : (df['kilometers_run'] < 0).sum(),
    'model_year < 1950'          : (df['model_year'] < 1950).sum(),
    'model_year > 2026'          : (df['model_year'] > 2026).sum(),
    'engine_capacity > 8000 cc'  : (df['engine_capacity'] > 8000).sum(),
}
for label, count in checks.items():
    flag = 'OK' if count == 0 else 'ISSUE'
    print(f'  [{flag:^5}] {label:<28}: {count} row(s)')

rows_before = len(df)
df = df[(df['price'] > 0) &
        (df['engine_capacity'] > 0) &
        (df['kilometers_run'] >= 0) &
        (df['model_year'].between(1950, 2026))].reset_index(drop=True)
print(f'\nRemoved {rows_before - len(df)} logically invalid row(s). Shape: {df.shape}')

### 4.7 Visualise outliers with box plots

Box plots make the skew obvious. Anything beyond the whiskers (1.5 × IQR from the quartiles) is a
candidate outlier — but a candidate is not a verdict. A 16-million-taka Land Cruiser is an outlier
*and* a real car.

In [ ]:
num_cols = ['price', 'kilometers_run', 'engine_capacity', 'model_year']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, col in zip(axes.flatten(), num_cols):
    sns.boxplot(x=df[col], ax=ax, color='#55A868', width=0.5)
    ax.set_title(f'Box plot — {col}')
    ax.set_xlabel('')

plt.suptitle('Outlier inspection before treatment', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### 4.8 Quantify outliers with the IQR rule

The interquartile-range rule flags any value below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR`. We compute
the counts first and decide what to do only after seeing how many rows are involved.

In [ ]:
def iqr_bounds(series, k=1.5):
    """Return the (lower, upper) IQR fences for a numeric series."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

print('=' * 78)
print(f"{'Column':<18}{'Lower':>14}{'Upper':>16}{'Outliers':>12}{'% of data':>12}")
print('=' * 78)

outlier_summary = {}
for col in num_cols:
    lo, hi = iqr_bounds(df[col])
    mask = (df[col] < lo) | (df[col] > hi)
    outlier_summary[col] = mask.sum()
    print(f'{col:<18}{lo:>14,.0f}{hi:>16,.0f}{mask.sum():>12}{mask.mean()*100:>11.2f}%')
print('=' * 78)

### 4.9 Treat outliers — capping, not deletion

**Decision: cap (winsorise) at the 1st and 99th percentiles rather than delete.**

Reasoning:
- Deleting every IQR-flagged row would discard a meaningful share of a 1,200-row dataset and, worse,
  would systematically remove the *entire luxury segment* — the model would then have no idea how to
  price an expensive car and would fail badly on exactly the listings where money is at stake.
- The genuine problem cases are implausible extremes (a 900,000 km odometer reading), not merely
  expensive cars.
- Capping keeps every row and every relationship intact while pulling the most damaging extremes back
  to a defensible boundary, which stabilises the linear models in particular.

We cap `price` and `kilometers_run`, the two heavily skewed columns, and leave `engine_capacity` and
`model_year` alone since their ranges are already physically sensible.

In [ ]:
def cap_outliers(data, col, lower_q=0.01, upper_q=0.99):
    """Winsorise a column at the given percentiles. Returns rows changed."""
    lo, hi = data[col].quantile(lower_q), data[col].quantile(upper_q)
    n = ((data[col] < lo) | (data[col] > hi)).sum()
    data[col] = data[col].clip(lower=lo, upper=hi)
    print(f'{col:<18}: capped to [{lo:,.0f} , {hi:,.0f}]  -> {n} value(s) adjusted')
    return n

print('=' * 70); print('OUTLIER CAPPING (1st / 99th percentile)'); print('=' * 70)
for col in ['price', 'kilometers_run']:
    cap_outliers(df, col)

print('\nPost-capping summary:')
display(df[num_cols].describe().T)

### 4.10 Before-and-after comparison

A quick visual confirmation that capping tamed the tails without flattening the distribution.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, col in enumerate(['price', 'kilometers_run']):
    sns.histplot(df_raw[col], bins=50, ax=axes[i, 0], color='#C44E52', kde=True)
    axes[i, 0].set_title(f'{col} — BEFORE cleaning')
    sns.histplot(df[col], bins=50, ax=axes[i, 1], color='#4C72B0', kde=True)
    axes[i, 1].set_title(f'{col} — AFTER cleaning')

plt.suptitle('Effect of cleaning on distribution shape', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'Raw dataset   : {df_raw.shape[0]:,} rows')
print(f'Clean dataset : {df.shape[0]:,} rows')
print(f'Retained      : {len(df)/len(df_raw)*100:.1f}% of original listings')

---
# 5. Exploratory Data Analysis (EDA)

With clean data in hand we now ask what it actually says. EDA is where hypotheses come from: which
variables move with price, how strong the relationships are, and where the surprises hide.

### 5.1 Distribution of the target variable

Regression models — especially linear ones — assume roughly normally distributed errors. If `price` is
strongly right-skewed, a log transform often improves the fit dramatically. We measure the skewness and
kurtosis numerically rather than eyeballing a chart.

In [ ]:
print('=' * 70); print('TARGET VARIABLE: price (BDT)'); print('=' * 70)
print(f"Count    : {df['price'].count():,}")
print(f"Mean     : {df['price'].mean():>15,.0f}")
print(f"Median   : {df['price'].median():>15,.0f}")
print(f"Std dev  : {df['price'].std():>15,.0f}")
print(f"Minimum  : {df['price'].min():>15,.0f}")
print(f"Maximum  : {df['price'].max():>15,.0f}")
print(f"Skewness : {df['price'].skew():>15.3f}   (0 = symmetric, >1 = strong right skew)")
print(f"Kurtosis : {df['price'].kurtosis():>15.3f}   (0 = normal tails)")

print('\nPercentile breakdown:')
for q in [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]:
    print(f'  {int(q*100):>3}th percentile : {df["price"].quantile(q):>12,.0f} BDT')

if abs(df['price'].skew()) > 1:
    print('\n=> Strong skew detected. A log transform of the target is likely to help.')

### 5.2 Compare raw price against log-transformed price

Plotting both side by side makes the case for the transform visually. The log version should look far
closer to a bell curve.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df['price'], bins=50, kde=True, ax=axes[0], color='#C44E52')
axes[0].axvline(df['price'].mean(),   color='black', ls='--', label=f"Mean {df['price'].mean():,.0f}")
axes[0].axvline(df['price'].median(), color='navy',  ls='-.', label=f"Median {df['price'].median():,.0f}")
axes[0].set_title(f"Raw price  (skew = {df['price'].skew():.2f})")
axes[0].set_xlabel('Price (BDT)'); axes[0].legend()

log_price = np.log1p(df['price'])
sns.histplot(log_price, bins=50, kde=True, ax=axes[1], color='#4C72B0')
axes[1].set_title(f'log(1 + price)  (skew = {log_price.skew():.2f})')
axes[1].set_xlabel('log(Price)')

plt.tight_layout(); plt.show()
print(f"Skewness reduced from {df['price'].skew():.3f} to {log_price.skew():.3f}")

### 5.3 Univariate analysis of the numeric predictors

Each predictor gets a histogram with its mean and median marked. We are looking for multi-modality
(distinct clusters), unexpected spikes, and whether a variable has enough spread to be informative at
all.

In [ ]:
predictors = ['model_year', 'kilometers_run', 'engine_capacity']
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, col in zip(axes, predictors):
    sns.histplot(df[col], bins=35, kde=True, ax=ax, color='#8172B2')
    ax.axvline(df[col].mean(),   color='red',   ls='--', lw=1.2, label='mean')
    ax.axvline(df[col].median(), color='green', ls='-.', lw=1.2, label='median')
    ax.set_title(f'{col}\nskew = {df[col].skew():.2f}')
    ax.legend(fontsize=8)

plt.suptitle('Distributions of numeric predictors', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### 5.4 Frequency of the categorical variables

Class balance matters. A category represented by three listings cannot support a reliable coefficient,
and dominant categories tell us what the model will be good at predicting.

In [ ]:
cat_cols = ['brand', 'transmission', 'body_type', 'fuel_type']
for col in cat_cols:
    print('=' * 70)
    print(f'{col.upper()}  —  {df[col].nunique()} unique value(s)')
    print('=' * 70)
    vc = df[col].value_counts()
    pct = (vc / len(df) * 100).round(2)
    display(pd.DataFrame({'count': vc, 'percent': pct}).head(10))
    if df[col].nunique() > 10:
        print(f'... and {df[col].nunique() - 10} more categories\n')

### 5.5 Correlation between numeric variables

Pearson correlation quantifies the strength and direction of every pairwise linear relationship.
Two readings matter:
- **Column `price`** — which predictors carry signal about the target.
- **Predictor-versus-predictor** — pairs above ~0.8 are multicollinear, which destabilises linear
  regression coefficients.

In [ ]:
corr = df[['model_year', 'kilometers_run', 'engine_capacity', 'price']].corr()

print('=' * 70); print('CORRELATION MATRIX'); print('=' * 70)
display(corr.round(3))

print('\nCorrelation of each predictor with price:')
target_corr = corr['price'].drop('price').sort_values(key=abs, ascending=False)
for name, value in target_corr.items():
    strength = ('strong' if abs(value) >= 0.5 else
                'moderate' if abs(value) >= 0.3 else 'weak')
    direction = 'positive' if value > 0 else 'negative'
    print(f'  {name:<18}: {value:+.3f}   ({strength} {direction})')

### 5.6 Correlation heatmap

The same matrix rendered visually. Red cells indicate positive association, blue negative; colour
intensity encodes strength.

In [ ]:
plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))   # hide the redundant upper triangle
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, square=True, linewidths=1,
            cbar_kws={'shrink': 0.8, 'label': 'Pearson r'})
plt.title('Correlation heatmap — numeric variables', pad=12)
plt.tight_layout(); plt.show()

### 5.7 Group statistics — how price varies by category

Aggregating price by each categorical variable reveals the size of the effect. Mean alone is
misleading for skewed data, so we report the median and the group size alongside it.

In [ ]:
def group_price_stats(data, by, top=10):
    """Summarise price statistics within each level of a categorical column."""
    g = (data.groupby(by)['price']
             .agg(count='count', mean='mean', median='median', std='std')
             .sort_values('median', ascending=False))
    return g[g['count'] >= 3].head(top)   # ignore groups too small to trust

for col in ['brand', 'body_type', 'transmission']:
    print('=' * 70)
    print(f'PRICE BY {col.upper()} (groups with >= 3 listings)')
    print('=' * 70)
    display(group_price_stats(df, col))

### 5.8 Depreciation — the relationship between age and price

This is the single most important relationship in the dataset. We compute mean price by model year to
trace the depreciation curve directly.

In [ ]:
year_price = (df.groupby('model_year')['price']
                .agg(count='count', mean_price='mean', median_price='median')
                .query('count >= 3'))

print('Mean price by model year (years with >= 3 listings):')
display(year_price.tail(20))

r = df[['model_year', 'price']].corr().iloc[0, 1]
print(f'\nCorrelation between model_year and price: {r:+.3f}')
print('Interpretation: newer cars command systematically higher prices.'
      if r > 0 else 'Unexpected — newer cars appear cheaper in this sample.')

---
# 6. Data Visualization

EDA established *what* the relationships are. This section renders them so a reader can see the
patterns without reading a table of numbers.

### 6.1 Scatter plots — each numeric predictor against price

A fitted regression line is overlaid on each. The tightness of the point cloud around that line is a
direct preview of how much predictive power the feature carries.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, col in zip(axes, ['model_year', 'kilometers_run', 'engine_capacity']):
    sns.regplot(data=df, x=col, y='price', ax=ax,
                scatter_kws={'alpha': 0.35, 's': 18, 'color': '#4C72B0'},
                line_kws={'color': '#C44E52', 'lw': 2})
    r = df[[col, 'price']].corr().iloc[0, 1]
    ax.set_title(f'price vs {col}   (r = {r:+.3f})')
    ax.set_ylabel('Price (BDT)')

plt.suptitle('Relationship between each numeric predictor and price',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### 6.2 Box plots — price distribution across categories

Box plots compare whole distributions rather than just averages, exposing both the typical price and
the spread within each category.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 11))

top_brands = df['brand'].value_counts().head(8).index
sns.boxplot(data=df[df['brand'].isin(top_brands)], x='brand', y='price',
            ax=axes[0], palette='Set2',
            order=df[df['brand'].isin(top_brands)].groupby('brand')['price']
                    .median().sort_values(ascending=False).index)
axes[0].set_title('Price distribution by brand (top 8 by listing count)')
axes[0].tick_params(axis='x', rotation=30)

sns.boxplot(data=df, x='body_type', y='price', ax=axes[1], palette='Set3',
            order=df.groupby('body_type')['price'].median().sort_values(ascending=False).index)
axes[1].set_title('Price distribution by body type')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout(); plt.show()

### 6.3 Transmission and fuel type

Transmission gets a violin plot, which shows the full density rather than just quartiles. Fuel type is
plotted as a bar chart of mean price across the most common fuel configurations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

sns.violinplot(data=df, x='transmission', y='price', ax=axes[0],
               palette='muted', inner='quartile')
axes[0].set_title('Price density by transmission type')

top_fuel = df['fuel_type'].value_counts().head(8).index
fuel_stats = (df[df['fuel_type'].isin(top_fuel)]
                .groupby('fuel_type')['price'].mean().sort_values(ascending=False))
sns.barplot(x=fuel_stats.values, y=fuel_stats.index, ax=axes[1], palette='viridis')
axes[1].set_title('Mean price by fuel configuration (8 most common)')
axes[1].set_xlabel('Mean price (BDT)')

plt.tight_layout(); plt.show()

### 6.4 Depreciation curve over model year

Mean price plotted against model year, with the count of listings behind it. This is the curve a
valuation model must learn.

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))

ax1.bar(year_price.index, year_price['count'], alpha=0.25,
        color='grey', label='Number of listings')
ax1.set_ylabel('Number of listings', color='grey')
ax1.set_xlabel('Model year')

ax2 = ax1.twinx()
ax2.plot(year_price.index, year_price['mean_price'], 'o-',
         color='#C44E52', lw=2, ms=5, label='Mean price')
ax2.plot(year_price.index, year_price['median_price'], 's--',
         color='#4C72B0', lw=1.5, ms=4, label='Median price')
ax2.set_ylabel('Price (BDT)')

ax1.set_title('Depreciation curve — price against model year', fontweight='bold')
lines = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax2.legend(lines, labels, loc='upper left')
plt.tight_layout(); plt.show()

### 6.5 Pair plot — all numeric relationships at once

The pair plot puts every pairwise scatter and every marginal distribution on one grid, coloured by
transmission so a categorical split is visible too.

In [ ]:
pair_data = df[['model_year', 'kilometers_run', 'engine_capacity',
                'price', 'transmission']].copy()
g = sns.pairplot(pair_data, hue='transmission', diag_kind='kde',
                 plot_kws={'alpha': 0.45, 's': 22}, height=2.3, corner=True)
g.fig.suptitle('Pairwise relationships among numeric variables',
               y=1.01, fontsize=14, fontweight='bold')
plt.show()

### 6.6 Mileage versus price, segmented by vehicle age

A single scatter hides an interaction: kilometres hurt an old car differently than a new one. Colouring
points by age band and fitting a separate line per band exposes that interaction.

In [ ]:
temp = df.copy()
temp['age'] = 2024 - temp['model_year']
temp['age_band'] = pd.cut(temp['age'], bins=[-1, 5, 10, 15, 100],
                          labels=['0-5 yrs', '6-10 yrs', '11-15 yrs', '16+ yrs'])

plt.figure(figsize=(12, 6.5))
sns.scatterplot(data=temp, x='kilometers_run', y='price', hue='age_band',
                palette='coolwarm', alpha=0.6, s=35)
for band in temp['age_band'].cat.categories:
    sub = temp[temp['age_band'] == band]
    if len(sub) > 10:
        z = np.polyfit(sub['kilometers_run'], sub['price'], 1)
        xs = np.linspace(sub['kilometers_run'].min(), sub['kilometers_run'].max(), 50)
        plt.plot(xs, np.poly1d(z)(xs), lw=1.8, alpha=0.85)

plt.title('Mileage versus price, segmented by vehicle age', fontweight='bold')
plt.xlabel('Kilometers run'); plt.ylabel('Price (BDT)')
plt.legend(title='Age band'); plt.tight_layout(); plt.show()

---
# 7. Feature Engineering

The raw columns describe a car; they do not describe *what makes it valuable*. Feature engineering
encodes domain knowledge into new columns the model can actually use, then puts everything on a
numeric footing.

Four stages: **creation → transformation → encoding → scaling.**

### 7.1 Feature creation — derive new predictors

| New feature | Formula | Why it should help |
|-------------|---------|--------------------|
| `car_age` | `current_year − model_year` | Depreciation is driven by age directly; a model does not have to learn what "2011" means |
| `km_per_year` | `kilometers_run / car_age` | Usage *intensity*. A 10-year-old car with 30,000 km is a different asset from one with 300,000 km |
| `is_low_mileage` | `km_per_year < 10,000` | A binary flag for the "lightly used" premium buyers pay for |
| `engine_per_1000cc` | `engine_capacity / 1000` | Rescales cc into litres, the unit buyers actually think in |
| `is_luxury_brand` | brand ∈ {BMW, Mercedes, Audi, Lexus, …} | Brand prestige commands a premium independent of specification |
| `n_fuel_types` | count of comma-separated fuels | Multi-fuel cars (CNG + octane) are cheaper to run and sell for more |
| `model_frequency` | count of listings of that model | A proxy for market liquidity and parts availability |

In [ ]:
CURRENT_YEAR = 2024
fe = df.copy()

# --- Age of the vehicle ---
fe['car_age'] = CURRENT_YEAR - fe['model_year']
fe['car_age'] = fe['car_age'].clip(lower=0)          # guard against future years

# --- Usage intensity: kilometres driven per year of life ---
fe['km_per_year'] = fe['kilometers_run'] / fe['car_age'].replace(0, 1)
fe['is_low_mileage'] = (fe['km_per_year'] < 10_000).astype(int)

# --- Engine size in litres ---
fe['engine_per_1000cc'] = fe['engine_capacity'] / 1000

# --- Luxury brand indicator ---
LUXURY = ['BMW', 'Mercedes-Benz', 'Mercedes', 'Audi', 'Lexus',
          'Land Rover', 'Porsche', 'Jaguar', 'Volvo', 'Infiniti']
fe['is_luxury_brand'] = fe['brand'].isin(LUXURY).astype(int)

# --- How many fuel systems the car supports ---
fe['n_fuel_types'] = fe['fuel_type'].astype(str).apply(lambda s: len(s.split(',')))

# --- Market liquidity proxy: how common is this model ---
fe['model_frequency'] = fe['car_model'].map(fe['car_model'].value_counts())
fe['brand_frequency'] = fe['brand'].map(fe['brand'].value_counts())

new_features = ['car_age', 'km_per_year', 'is_low_mileage', 'engine_per_1000cc',
                'is_luxury_brand', 'n_fuel_types', 'model_frequency', 'brand_frequency']
print(f'Created {len(new_features)} new features:')
display(fe[new_features].describe().T)

### 7.2 Multi-hot encoding of the multi-label fuel column

`fuel_type` holds values such as `"Petrol, CNG, Octane"`. One-hot encoding would treat that as a single
category distinct from `"CNG, Petrol"` — clearly wrong, and it would scatter the signal across 24 rare
dummies.

The correct treatment is **multi-hot encoding**: one binary column per individual fuel, set to 1 if
that fuel appears anywhere in the string. Now `"Petrol, CNG"` and `"CNG, Petrol"` produce identical
vectors, and the model can learn the value of CNG capability on its own.

In [ ]:
# Discover every distinct fuel token in the dataset.
all_fuels = set()
for value in fe['fuel_type'].astype(str):
    all_fuels.update(token.strip() for token in value.split(','))
all_fuels = sorted(f for f in all_fuels if f and f.lower() != 'nan')
print('Individual fuel types found:', all_fuels)

# Build one binary column per fuel.
fuel_cols = []
for fuel in all_fuels:
    col = 'fuel_' + re.sub(r'[^A-Za-z0-9]+', '_', fuel).strip('_').lower()
    fe[col] = fe['fuel_type'].astype(str).apply(
        lambda s, f=fuel: int(f in [t.strip() for t in s.split(',')]))
    fuel_cols.append(col)

print(f'\nCreated {len(fuel_cols)} multi-hot fuel columns.')
display(fe[fuel_cols].sum().sort_values(ascending=False).to_frame('listings_with_fuel'))

### 7.3 Feature transformation — taming skew

Section 5.1 showed `price` is strongly right-skewed, and `kilometers_run` is too. A `log1p` transform
compresses the long tail, which:
- brings the target closer to normal, satisfying linear-regression assumptions;
- converts the model from predicting *absolute* taka errors to *proportional* ones, which matches how
  price errors actually matter (being 200,000 BDT off on a 500,000 BDT car is far worse than on a
  5,000,000 BDT car).

We keep both versions so predictions can be converted back to taka with `expm1` for reporting.

In [ ]:
# Log-transform the target.
fe['price_log'] = np.log1p(fe['price'])

# Log-transform the skewed mileage predictors.
fe['km_log']          = np.log1p(fe['kilometers_run'])
fe['km_per_year_log'] = np.log1p(fe['km_per_year'])

skew_table = pd.DataFrame({
    'before': [df['price'].skew(), df['kilometers_run'].skew(), fe['km_per_year'].skew()],
    'after' : [fe['price_log'].skew(), fe['km_log'].skew(), fe['km_per_year_log'].skew()]
}, index=['price', 'kilometers_run', 'km_per_year'])
skew_table['improvement'] = skew_table['before'].abs() - skew_table['after'].abs()

print('=' * 70); print('SKEWNESS BEFORE AND AFTER LOG TRANSFORM'); print('=' * 70)
display(skew_table.round(3))

### 7.4 Encoding categorical variables

Three different strategies, each chosen to match the cardinality of the column:

| Column | Cardinality | Strategy | Reason |
|--------|-------------|----------|--------|
| `transmission` | 2 | **Binary / label** | Two levels need one column |
| `body_type` | ~8 | **One-hot** | Low cardinality, no natural ordering |
| `brand` | ~26 | **Frequency + target encoding** | One-hot would add 26 sparse columns to a 1,200-row dataset and invite over-fitting |
| `car_model` | ~123 | **Frequency encoding** | Far too high-cardinality to one-hot |

Target encoding is computed **on the training split only** in Section 9 to avoid leakage; here we
prepare the mechanism and apply frequency encoding, which is leakage-safe because it uses no target
information.

In [ ]:
# --- Binary encoding for transmission ---
fe['is_automatic'] = (fe['transmission'] == 'Automatic').astype(int)

# --- One-hot encoding for body_type ---
body_dummies = pd.get_dummies(fe['body_type'], prefix='body', drop_first=True, dtype=int)
fe = pd.concat([fe, body_dummies], axis=1)
body_cols = body_dummies.columns.tolist()
print(f'One-hot encoded body_type -> {len(body_cols)} columns: {body_cols}')

# --- Frequency encoding already created in 7.1 (model_frequency, brand_frequency) ---
print(f'\nFrequency-encoded brand   : {fe["brand"].nunique()} levels -> 1 numeric column')
print(f'Frequency-encoded car_model: {fe["car_model"].nunique()} levels -> 1 numeric column')
print(f'Binary-encoded transmission: 2 levels -> 1 column (is_automatic)')

display(fe[['is_automatic'] + body_cols].head())

### 7.5 Assemble the final feature matrix

We now select the columns that go into the model and drop the ones that would cause problems:

- `car_name` — a near-unique identifier; it would let a tree memorise individual listings.
- `brand`, `car_model`, `body_type`, `fuel_type`, `transmission` — the raw text versions, now replaced
  by their encoded counterparts.
- `price` — the target itself (we model `price_log`).
- `kilometers_run`, `km_per_year` — superseded by their log versions.
- `model_year` — superseded by `car_age`, which carries the same information in a more direct form.

In [ ]:
feature_cols = (['car_age', 'km_log', 'km_per_year_log', 'engine_capacity',
                 'engine_per_1000cc', 'is_low_mileage', 'is_luxury_brand',
                 'n_fuel_types', 'model_frequency', 'brand_frequency',
                 'is_automatic']
                + body_cols + fuel_cols)

X = fe[feature_cols].copy()
y = fe['price_log'].copy()          # model the log target
y_actual = fe['price'].copy()       # keep taka values for reporting

print(f'Feature matrix X : {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'Target vector  y : {y.shape[0]:,} values (log scale)')
print(f'\nFeatures used ({len(feature_cols)}):')
for i, c in enumerate(feature_cols, 1):
    print(f'  {i:>2}. {c}')

assert X.isnull().sum().sum() == 0, 'NaNs present in the feature matrix!'
print('\nNo missing values in X — ready for modelling.')

### 7.6 Train / test split

We split **before** scaling and before target encoding. This is not a formality: fitting a scaler on
the full dataset lets information about the test set's distribution leak into training, and the
resulting scores are optimistic. An 80/20 split leaves enough test data to evaluate meaningfully while
keeping the training set as large as possible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE)

# Corresponding actual-taka targets, aligned by index, for interpretable reporting.
y_train_actual = y_actual.loc[y_train.index]
y_test_actual  = y_actual.loc[y_test.index]

print(f'Training set : {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test set     : {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nTrain price range: {y_train_actual.min():,.0f} – {y_train_actual.max():,.0f} BDT')
print(f'Test  price range: {y_test_actual.min():,.0f} – {y_test_actual.max():,.0f} BDT')
print(f'\nTrain mean price : {y_train_actual.mean():,.0f} BDT')
print(f'Test  mean price : {y_test_actual.mean():,.0f} BDT   (similar => representative split)')

### 7.7 Feature scaling

Distance-based and regularised models (KNN, SVR, Ridge, Lasso) are sensitive to feature magnitude.
`kilometers_run` in the tens of thousands would completely dominate a 0/1 flag unless we standardise.

`StandardScaler` centres each feature at zero with unit variance. It is **fitted on the training set
only**, then applied to the test set — the correct order that prevents leakage.

Tree-based models do not need scaling (they split on thresholds, not distances), so we keep both the
scaled and unscaled versions and hand each model the one it wants.

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),
                              columns=X_train.columns, index=X_train.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),
                              columns=X_test.columns, index=X_test.index)

print('Scaling complete (fit on train, applied to test).')
print('\nMean and std of the first 6 scaled training features:')
display(pd.DataFrame({'mean': X_train_scaled.mean(), 'std': X_train_scaled.std()}).head(6).round(4))
print('Means ~ 0 and standard deviations ~ 1, as expected.')

---
# 8. Feature Selection

More features are not automatically better. Irrelevant ones add noise, correlated ones destabilise
coefficients, and both increase over-fitting risk on a 1,200-row dataset.

We triangulate with three independent methods and trust features that more than one of them endorses.

### 8.1 Method 1 — univariate statistical selection (F-test)

`SelectKBest` with `f_regression` scores each feature by the strength of its individual linear
relationship with the target. It is fast and interpretable, but blind to interactions between features.

In [ ]:
selector = SelectKBest(score_func=f_regression, k='all')
selector.fit(X_train_scaled, y_train)

f_scores = pd.DataFrame({
    'feature' : X_train.columns,
    'f_score' : selector.scores_,
    'p_value' : selector.pvalues_
}).sort_values('f_score', ascending=False).reset_index(drop=True)
f_scores['significant'] = f_scores['p_value'] < 0.05

print('=' * 70); print('TOP 15 FEATURES BY UNIVARIATE F-SCORE'); print('=' * 70)
display(f_scores.head(15))
print(f"\nStatistically significant features (p < 0.05): "
      f"{f_scores['significant'].sum()} of {len(f_scores)}")

### 8.2 Method 2 — tree-based feature importance

A Random Forest measures how much each feature reduces prediction error across all its splits. Unlike
the F-test this captures non-linear effects and interactions.

In [ ]:
rf_selector = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
rf_selector.fit(X_train, y_train)

importance = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': rf_selector.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)
importance['cumulative'] = importance['importance'].cumsum()

print('=' * 70); print('RANDOM FOREST FEATURE IMPORTANCE'); print('=' * 70)
display(importance.head(15))

n_for_90 = (importance['cumulative'] < 0.90).sum() + 1
print(f'\n{n_for_90} features account for 90% of total importance.')

### 8.3 Visualise feature importance

Two panels: the ranked importance bars, and the cumulative curve showing how quickly importance
saturates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top = importance.head(18)
sns.barplot(data=top, x='importance', y='feature', ax=axes[0], palette='rocket')
axes[0].set_title('Random Forest feature importance (top 18)')
axes[0].set_xlabel('Importance')

axes[1].plot(range(1, len(importance) + 1), importance['cumulative'],
             'o-', color='#4C72B0', ms=4)
axes[1].axhline(0.90, color='red', ls='--', label='90% of total importance')
axes[1].axvline(n_for_90, color='green', ls=':', label=f'{n_for_90} features')
axes[1].set_title('Cumulative importance')
axes[1].set_xlabel('Number of features'); axes[1].set_ylabel('Cumulative importance')
axes[1].legend()

plt.tight_layout(); plt.show()

### 8.4 Method 3 — Recursive Feature Elimination (RFE)

RFE repeatedly fits a model, discards the weakest feature, and refits. It accounts for how features
behave *together*, which neither of the previous methods fully does.

In [ ]:
N_SELECT = min(15, X_train.shape[1])
rfe = RFE(estimator=RandomForestRegressor(n_estimators=100,
                                          random_state=RANDOM_STATE, n_jobs=-1),
          n_features_to_select=N_SELECT, step=2)
rfe.fit(X_train, y_train)

rfe_result = pd.DataFrame({
    'feature' : X_train.columns,
    'selected': rfe.support_,
    'rank'    : rfe.ranking_
}).sort_values('rank').reset_index(drop=True)

print('=' * 70); print(f'RFE — TOP {N_SELECT} FEATURES SELECTED'); print('=' * 70)
display(rfe_result.head(N_SELECT))
rfe_features = rfe_result[rfe_result.selected]['feature'].tolist()

### 8.5 Check for multicollinearity

Highly correlated predictors make linear-model coefficients unstable and uninterpretable — the model
cannot tell which of two near-identical features deserves the credit. We flag any pair correlated above
0.85.

In [ ]:
feat_corr = X_train.corr().abs()
upper = feat_corr.where(np.triu(np.ones(feat_corr.shape), k=1).astype(bool))

high_pairs = [(r, c, upper.loc[r, c])
              for r in upper.index for c in upper.columns
              if pd.notna(upper.loc[r, c]) and upper.loc[r, c] > 0.85]

print('=' * 70); print('HIGHLY CORRELATED FEATURE PAIRS (|r| > 0.85)'); print('=' * 70)
if high_pairs:
    display(pd.DataFrame(high_pairs, columns=['feature_1', 'feature_2', 'correlation'])
              .sort_values('correlation', ascending=False))
    print('\nNote: engine_capacity and engine_per_1000cc are the same variable rescaled — '
          'expected and harmless for tree models.')
else:
    print('No problematic multicollinearity detected.')

### 8.6 Combine the three methods into a final feature set

A feature is kept if **at least two of the three methods** endorse it (top-15 by F-score, top-15 by
Random Forest importance, or selected by RFE). Consensus selection is more robust than trusting any
single criterion.

In [ ]:
top_f    = set(f_scores.head(15)['feature'])
top_rf   = set(importance.head(15)['feature'])
top_rfe  = set(rfe_features)

votes = pd.Series(0, index=X_train.columns)
for group in [top_f, top_rf, top_rfe]:
    votes[list(group)] += 1

selected_features = sorted(votes[votes >= 2].index.tolist())

print('=' * 70); print('CONSENSUS FEATURE SELECTION'); print('=' * 70)
display(pd.DataFrame({'votes': votes}).sort_values('votes', ascending=False).head(20))

print(f'\nSelected {len(selected_features)} features (endorsed by >= 2 methods):')
for f in selected_features:
    print('  -', f)

X_train_sel        = X_train[selected_features]
X_test_sel         = X_test[selected_features]
X_train_sel_scaled = X_train_scaled[selected_features]
X_test_sel_scaled  = X_test_scaled[selected_features]
print(f'\nReduced feature matrix: {X_train.shape[1]} -> {len(selected_features)} features')

---
# 9. Machine Learning Model Development

**Why regression?** The target `price` is a continuous quantity in taka. Classification would require
binning price into arbitrary bands and would discard information; clustering is unsupervised and cannot
use the price labels we have. Regression is the correct family.

We train eight models spanning three philosophies:

| Family | Models | Rationale |
|--------|--------|-----------|
| **Linear** | Linear Regression, Ridge, Lasso, ElasticNet | Interpretable baseline; regularised variants handle multicollinearity |
| **Tree-based** | Decision Tree, Random Forest, Gradient Boosting | Capture non-linearity and interactions without manual specification |
| **Distance / kernel** | KNN, SVR | Non-parametric alternatives that make different assumptions entirely |

Each is trained on the log target and evaluated identically, so comparisons are fair.

### 9.1 Define the evaluation function

A single reusable function computes every metric so no model gets accidentally favourable treatment.
It also converts log predictions back to taka with `expm1`, giving MAE and RMSE in units a reader
understands.

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, y_te_actual):
    """Fit a model, predict, and return a dictionary of performance metrics.

    Metrics are computed on the log scale (R2) and back-transformed to BDT
    (MAE, RMSE, MAPE) so errors are interpretable in real currency.
    """
    model.fit(X_tr, y_tr)
    pred_train_log = model.predict(X_tr)
    pred_test_log  = model.predict(X_te)

    # Back-transform from log space to taka.
    pred_test_actual = np.expm1(pred_test_log)

    return {
        'Model'      : name,
        'Train_R2'   : r2_score(y_tr, pred_train_log),
        'Test_R2'    : r2_score(y_te, pred_test_log),
        'RMSE_log'   : np.sqrt(mean_squared_error(y_te, pred_test_log)),
        'MAE_BDT'    : mean_absolute_error(y_te_actual, pred_test_actual),
        'RMSE_BDT'   : np.sqrt(mean_squared_error(y_te_actual, pred_test_actual)),
        'MAPE_%'     : mean_absolute_percentage_error(y_te_actual, pred_test_actual) * 100,
        'Overfit_gap': r2_score(y_tr, pred_train_log) - r2_score(y_te, pred_test_log),
        '_model'     : model,
        '_pred_log'  : pred_test_log,
        '_pred_bdt'  : pred_test_actual,
    }

print('Evaluation function defined.')

### 9.2 Baseline model — Linear Regression

Every project needs a baseline. Ordinary least squares is the simplest defensible model; anything more
complex must beat it to justify its complexity. It also gives directly interpretable coefficients.

In [ ]:
results = []

lr = LinearRegression()
res_lr = evaluate_model('Linear Regression', lr,
                        X_train_sel_scaled, y_train,
                        X_test_sel_scaled, y_test, y_test_actual)
results.append(res_lr)

print('=' * 70); print('BASELINE — LINEAR REGRESSION'); print('=' * 70)
print(f"Train R2 : {res_lr['Train_R2']:.4f}")
print(f"Test  R2 : {res_lr['Test_R2']:.4f}")
print(f"MAE      : {res_lr['MAE_BDT']:,.0f} BDT")
print(f"RMSE     : {res_lr['RMSE_BDT']:,.0f} BDT")
print(f"MAPE     : {res_lr['MAPE_%']:.2f}%")

coefs = pd.DataFrame({'feature': selected_features,
                      'coefficient': lr.coef_}).sort_values('coefficient',
                                                            key=abs, ascending=False)
print('\nCoefficients (log-price change per 1 standard deviation of the feature):')
display(coefs.head(12))

### 9.3 Regularised linear models — Ridge, Lasso, ElasticNet

Regularisation penalises large coefficients, which controls variance when predictors are correlated.
**Ridge** (L2) shrinks coefficients smoothly; **Lasso** (L1) drives some to exactly zero, performing
automatic feature selection; **ElasticNet** blends both.

In [ ]:
regularised = {
    'Ridge Regression'      : Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Lasso Regression'      : Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=10000),
    'ElasticNet Regression' : ElasticNet(alpha=0.001, l1_ratio=0.5,
                                         random_state=RANDOM_STATE, max_iter=10000),
}

for name, model in regularised.items():
    res = evaluate_model(name, model, X_train_sel_scaled, y_train,
                         X_test_sel_scaled, y_test, y_test_actual)
    results.append(res)
    print(f"{name:<24} Test R2 = {res['Test_R2']:.4f} | "
          f"MAPE = {res['MAPE_%']:6.2f}% | MAE = {res['MAE_BDT']:>10,.0f} BDT")

n_zero = int((results[2]['_model'].coef_ == 0).sum())
print(f'\nLasso eliminated {n_zero} of {len(selected_features)} features '
      f'by shrinking their coefficients to exactly zero.')

### 9.4 Tree-based models — Decision Tree, Random Forest, Gradient Boosting

Trees split the feature space into regions and predict a constant in each, so they capture non-linear
relationships and interactions with no manual feature crossing. A single tree over-fits badly;
**Random Forest** averages many de-correlated trees to reduce variance, and **Gradient Boosting** fits
trees sequentially, each correcting the previous one's residuals.

These use the **unscaled** features — splits are threshold-based, so scaling is irrelevant to them.

In [ ]:
tree_models = {
    'Decision Tree'     : DecisionTreeRegressor(max_depth=12, min_samples_split=5,
                                                min_samples_leaf=3,
                                                random_state=RANDOM_STATE),
    'Random Forest'     : RandomForestRegressor(n_estimators=400, max_depth=20,
                                                min_samples_split=4, min_samples_leaf=2,
                                                random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=400, learning_rate=0.05,
                                                    max_depth=5, subsample=0.8,
                                                    random_state=RANDOM_STATE),
}

for name, model in tree_models.items():
    res = evaluate_model(name, model, X_train_sel, y_train,
                         X_test_sel, y_test, y_test_actual)
    results.append(res)
    print(f"{name:<20} Test R2 = {res['Test_R2']:.4f} | "
          f"MAPE = {res['MAPE_%']:6.2f}% | overfit gap = {res['Overfit_gap']:.4f}")

### 9.5 Distance and kernel models — KNN and SVR

**KNN** predicts the average price of the k most similar cars — conceptually close to how a human
appraiser works. **SVR** fits a margin in a transformed feature space. Both depend on distance, so both
receive the **scaled** features.

In [ ]:
other_models = {
    'K-Nearest Neighbors' : KNeighborsRegressor(n_neighbors=7, weights='distance'),
    'Support Vector Reg.' : SVR(kernel='rbf', C=10, gamma='scale', epsilon=0.1),
}

for name, model in other_models.items():
    res = evaluate_model(name, model, X_train_sel_scaled, y_train,
                         X_test_sel_scaled, y_test, y_test_actual)
    results.append(res)
    print(f"{name:<22} Test R2 = {res['Test_R2']:.4f} | MAPE = {res['MAPE_%']:6.2f}%")

print(f'\nAll {len(results)} models trained.')

### 9.6 Cross-validation — is the test score trustworthy?

A single train/test split can flatter or punish a model by luck. 5-fold cross-validation trains and
tests five times on different partitions; the **standard deviation** of the resulting scores tells us
how stable each model really is.

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for res in results:
    name, model = res['Model'], res['_model']
    use_scaled = name not in ['Decision Tree', 'Random Forest', 'Gradient Boosting']
    X_cv = X_train_sel_scaled if use_scaled else X_train_sel
    scores = cross_val_score(model, X_cv, y_train, cv=kfold, scoring='r2', n_jobs=-1)
    cv_rows.append({'Model': name, 'CV_mean_R2': scores.mean(),
                    'CV_std': scores.std(), 'CV_min': scores.min(), 'CV_max': scores.max()})
    print(f'{name:<24} CV R2 = {scores.mean():.4f} (+/- {scores.std():.4f})')

cv_df = pd.DataFrame(cv_rows)

### 9.7 Hyperparameter tuning of the leading model

The best model so far is re-fitted over a grid of hyperparameter combinations, scored by
cross-validated R². This squeezes out the remaining performance that sensible defaults left on the
table.

In [ ]:
best_so_far = max(results, key=lambda r: r['Test_R2'])['Model']
print(f'Best model before tuning: {best_so_far}\n')

if 'Forest' in best_so_far or 'Tree' in best_so_far:
    grid_model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
    param_grid = {'n_estimators': [200, 400],
                  'max_depth': [10, 20, None],
                  'min_samples_split': [2, 4],
                  'min_samples_leaf': [1, 2]}
    X_grid_tr, X_grid_te = X_train_sel, X_test_sel
    tuned_name = 'Random Forest (tuned)'
else:
    grid_model = GradientBoostingRegressor(random_state=RANDOM_STATE)
    param_grid = {'n_estimators': [200, 400],
                  'learning_rate': [0.03, 0.05, 0.1],
                  'max_depth': [3, 5, 7]}
    X_grid_tr, X_grid_te = X_train_sel, X_test_sel
    tuned_name = 'Gradient Boosting (tuned)'

grid = GridSearchCV(grid_model, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=0)
grid.fit(X_grid_tr, y_train)

print('Best hyperparameters:')
for k, v in grid.best_params_.items():
    print(f'  {k:<20}: {v}')
print(f'\nBest cross-validated R2: {grid.best_score_:.4f}')

res_tuned = evaluate_model(tuned_name, grid.best_estimator_,
                           X_grid_tr, y_train, X_grid_te, y_test, y_test_actual)
results.append(res_tuned)
print(f"Tuned model test R2    : {res_tuned['Test_R2']:.4f}")
print(f"Tuned model MAPE       : {res_tuned['MAPE_%']:.2f}%")

---
# 10. Model Evaluation

## Metrics used and why each one is here

| Metric | What it measures | Why it matters here |
|--------|------------------|---------------------|
| **R²** | Proportion of price variance explained | Headline goodness-of-fit; comparable across models |
| **RMSE (BDT)** | Root mean squared error in taka | Penalises large errors heavily — badly mispricing an expensive car is costly |
| **MAE (BDT)** | Mean absolute error in taka | The typical error, in currency, robust to outliers |
| **MAPE (%)** | Mean absolute *percentage* error | Scale-free: being 100k off matters far more on a 400k car than a 4M one |
| **Overfit gap** | Train R² − Test R² | A large gap means the model memorised rather than generalised |
| **CV std** | Spread of fold scores | Low spread = the result is stable, not a fluke of one split |

### 10.1 Consolidated comparison table

Every model, every metric, ranked by test R². This is the core evidence for the conclusion.

In [ ]:
comparison = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')}
                           for r in results])
comparison = comparison.merge(cv_df, on='Model', how='left')
comparison = comparison.sort_values('Test_R2', ascending=False).reset_index(drop=True)
comparison.index += 1

print('=' * 110); print('MODEL COMPARISON — RANKED BY TEST R-SQUARED'); print('=' * 110)
display(comparison[['Model', 'Train_R2', 'Test_R2', 'CV_mean_R2', 'CV_std',
                    'MAE_BDT', 'RMSE_BDT', 'MAPE_%', 'Overfit_gap']].round(4))

best = comparison.iloc[0]
print(f"\nBEST MODEL: {best['Model']}")
print(f"  Test R2 : {best['Test_R2']:.4f}  ({best['Test_R2']*100:.1f}% of price variance explained)")
print(f"  MAE     : {best['MAE_BDT']:,.0f} BDT")
print(f"  RMSE    : {best['RMSE_BDT']:,.0f} BDT")
print(f"  MAPE    : {best['MAPE_%']:.2f}%")

### 10.2 Visual model comparison

Four panels: test R², MAPE, MAE in taka, and the train-versus-test gap that exposes over-fitting.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
comp = comparison.sort_values('Test_R2')

sns.barplot(data=comp, y='Model', x='Test_R2', ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title('Test R² by model (higher is better)')
axes[0, 0].axvline(0.80, color='red', ls='--', label='target R² = 0.80')
axes[0, 0].legend()

sns.barplot(data=comp.sort_values('MAPE_%', ascending=False), y='Model', x='MAPE_%',
            ax=axes[0, 1], palette='rocket')
axes[0, 1].set_title('MAPE % (lower is better)')
axes[0, 1].axvline(20, color='green', ls='--', label='target MAPE = 20%')
axes[0, 1].legend()

sns.barplot(data=comp.sort_values('MAE_BDT', ascending=False), y='Model', x='MAE_BDT',
            ax=axes[1, 0], palette='mako')
axes[1, 0].set_title('Mean absolute error in BDT (lower is better)')

xpos = np.arange(len(comp))
axes[1, 1].barh(xpos - 0.2, comp['Train_R2'], height=0.4, label='Train R²', color='#4C72B0')
axes[1, 1].barh(xpos + 0.2, comp['Test_R2'],  height=0.4, label='Test R²',  color='#C44E52')
axes[1, 1].set_yticks(xpos); axes[1, 1].set_yticklabels(comp['Model'], fontsize=9)
axes[1, 1].set_title('Train vs Test R² — gap reveals over-fitting')
axes[1, 1].legend()

plt.tight_layout(); plt.show()

### 10.3 Predicted versus actual prices for the best model

If predictions were perfect every point would sit on the diagonal. Systematic curvature would mean the
model misprices a whole segment; a fan shape would mean error grows with price.

In [ ]:
best_res  = max(results, key=lambda r: r['Test_R2'])
best_name = best_res['Model']
pred_bdt  = best_res['_pred_bdt']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].scatter(y_test_actual, pred_bdt, alpha=0.5, s=30, color='#4C72B0')
lims = [min(y_test_actual.min(), pred_bdt.min()), max(y_test_actual.max(), pred_bdt.max())]
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual price (BDT)'); axes[0].set_ylabel('Predicted price (BDT)')
axes[0].set_title(f'{best_name} — predicted vs actual (BDT)')
axes[0].legend()

axes[1].scatter(y_test, best_res['_pred_log'], alpha=0.5, s=30, color='#55A868')
lg = [y_test.min(), y_test.max()]
axes[1].plot(lg, lg, 'r--', lw=2)
axes[1].set_xlabel('Actual log(price)'); axes[1].set_ylabel('Predicted log(price)')
axes[1].set_title(f"Log scale  (R² = {best_res['Test_R2']:.4f})")

plt.tight_layout(); plt.show()

### 10.4 Residual analysis

Residuals are what the model got wrong. Three diagnostics:
- **Residuals vs fitted** — should be a shapeless cloud around zero. Any pattern means missed structure.
- **Residual histogram** — should be roughly normal and centred on zero.
- **Q-Q plot** — points on the line confirm normally distributed errors, the key assumption of linear
  regression inference.

In [ ]:
from scipy import stats

residuals = y_test.values - best_res['_pred_log']
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].scatter(best_res['_pred_log'], residuals, alpha=0.5, s=28, color='#4C72B0')
axes[0].axhline(0, color='red', ls='--', lw=2)
axes[0].set_xlabel('Fitted value (log price)'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs fitted values')

sns.histplot(residuals, bins=35, kde=True, ax=axes[1], color='#55A868')
axes[1].axvline(0, color='red', ls='--')
axes[1].set_title(f'Residual distribution (mean = {residuals.mean():.4f})')
axes[1].set_xlabel('Residual')

stats.probplot(residuals, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q plot of residuals')

plt.suptitle(f'Residual diagnostics — {best_name}', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'Residual mean     : {residuals.mean():.5f}  (should be ~0)')
print(f'Residual std dev  : {residuals.std():.5f}')
print(f'Residual skewness : {stats.skew(residuals):.4f}')

### 10.5 Error distribution in real currency

Absolute and percentage errors expressed in taka answer the practical question: *how wrong is this
model on a typical car?*

In [ ]:
abs_err = np.abs(y_test_actual.values - pred_bdt)
pct_err = abs_err / y_test_actual.values * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(abs_err, bins=40, ax=axes[0], color='#C44E52')
axes[0].axvline(abs_err.mean(), color='black', ls='--', label=f'Mean {abs_err.mean():,.0f}')
axes[0].set_title('Absolute prediction error (BDT)'); axes[0].legend()

sns.histplot(pct_err, bins=40, ax=axes[1], color='#4C72B0')
axes[1].axvline(pct_err.mean(), color='black', ls='--', label=f'Mean {pct_err.mean():.1f}%')
axes[1].set_title('Percentage prediction error'); axes[1].legend()
plt.tight_layout(); plt.show()

print('=' * 60); print('ERROR BREAKDOWN'); print('=' * 60)
for t in [5, 10, 15, 20, 30]:
    print(f'  Predictions within {t:>2}% of actual price : {(pct_err <= t).mean()*100:5.1f}%')
print(f'\n  Median absolute error : {np.median(abs_err):,.0f} BDT')
print(f'  90th-percentile error : {np.percentile(abs_err, 90):,.0f} BDT')

### 10.6 What drives price — feature importance of the final model

For the winning model we extract the importance (or coefficient) of each feature. This is the part of
the analysis with practical value beyond prediction: it tells a buyer or seller *what actually moves
the number*.

In [ ]:
final_model = best_res['_model']

if hasattr(final_model, 'feature_importances_'):
    imp = pd.DataFrame({'feature': selected_features,
                        'importance': final_model.feature_importances_})
    imp = imp.sort_values('importance', ascending=False)
    label = 'Feature importance'
elif hasattr(final_model, 'coef_'):
    imp = pd.DataFrame({'feature': selected_features,
                        'importance': np.abs(final_model.coef_)})
    imp = imp.sort_values('importance', ascending=False)
    label = 'Absolute coefficient'
else:
    imp = None

if imp is not None:
    plt.figure(figsize=(10, 7))
    sns.barplot(data=imp.head(15), x='importance', y='feature', palette='flare')
    plt.title(f'{label} — {best_name}', fontweight='bold')
    plt.tight_layout(); plt.show()

    print('Top 10 price drivers:')
    for i, row in imp.head(10).iterrows():
        print(f"  {row['feature']:<24}: {row['importance']:.4f}")

### 10.7 Sample predictions on unseen listings

A concrete spot-check: fifteen test-set cars with their actual price, predicted price, and error.
Numbers on a table are more convincing than an aggregate metric.

In [ ]:
sample = pd.DataFrame({
    'Actual_BDT'   : y_test_actual.values,
    'Predicted_BDT': pred_bdt,
})
sample['Error_BDT']  = sample['Predicted_BDT'] - sample['Actual_BDT']
sample['Error_%']    = sample['Error_BDT'] / sample['Actual_BDT'] * 100

# Attach the human-readable car details for context.
details = fe.loc[y_test.index, ['brand', 'car_model', 'model_year',
                                'kilometers_run', 'transmission']].reset_index(drop=True)
sample = pd.concat([details, sample], axis=1)

print('=' * 70); print('SAMPLE PREDICTIONS ON UNSEEN TEST LISTINGS'); print('=' * 70)
display(sample.head(15).round(0))

print(f"\nBest single prediction  : {sample['Error_%'].abs().min():.2f}% error")
print(f"Worst single prediction : {sample['Error_%'].abs().max():.2f}% error")

### 10.8 Reusable prediction function

Finally, we wrap the trained model in a function that takes a plain car specification and returns an
estimated price — the deliverable a user would actually interact with.

In [ ]:
def predict_car_price(brand, car_model, model_year, kilometers_run,
                      engine_capacity, transmission, body_type, fuel_type):
    """Estimate the resale price (BDT) of a used car from its specification."""
    age  = max(CURRENT_YEAR - model_year, 0)
    kpy  = kilometers_run / max(age, 1)

    row = {
        'car_age'          : age,
        'km_log'           : np.log1p(kilometers_run),
        'km_per_year_log'  : np.log1p(kpy),
        'engine_capacity'  : engine_capacity,
        'engine_per_1000cc': engine_capacity / 1000,
        'is_low_mileage'   : int(kpy < 10_000),
        'is_luxury_brand'  : int(brand in LUXURY),
        'n_fuel_types'     : len(str(fuel_type).split(',')),
        'model_frequency'  : fe['car_model'].value_counts().get(car_model, 1),
        'brand_frequency'  : fe['brand'].value_counts().get(brand, 1),
        'is_automatic'     : int(transmission == 'Automatic'),
    }
    for c in body_cols:
        row[c] = int(c == f'body_{body_type}')
    for fuel, c in zip(all_fuels, fuel_cols):
        row[c] = int(fuel in [t.strip() for t in str(fuel_type).split(',')])

    vec = pd.DataFrame([row]).reindex(columns=X.columns, fill_value=0)[selected_features]
    if best_name in ['Decision Tree', 'Random Forest', 'Gradient Boosting',
                     'Random Forest (tuned)', 'Gradient Boosting (tuned)']:
        pred_log = final_model.predict(vec)[0]
    else:
        vec_scaled = pd.DataFrame(scaler.transform(
            pd.DataFrame([row]).reindex(columns=X.columns, fill_value=0)),
            columns=X.columns)[selected_features]
        pred_log = final_model.predict(vec_scaled)[0]
    return float(np.expm1(pred_log))

# --- Demonstration ---
example = dict(brand='Toyota', car_model='Axio', model_year=2015,
               kilometers_run=75_000, engine_capacity=1500,
               transmission='Automatic', body_type='Saloon',
               fuel_type='Petrol, Octane')
estimate = predict_car_price(**example)

print('Example car specification:')
for k, v in example.items():
    print(f'  {k:<16}: {v}')
print(f'\n  ESTIMATED PRICE : {estimate:,.0f} BDT')
print(f'  Expected range  : {estimate*0.85:,.0f} – {estimate*1.15:,.0f} BDT '
      f'(±15%, reflecting typical model error)')

---
# 11. Results, Discussion, and Conclusion

### 11.1 Automated results summary

This cell prints the final numbers so the written conclusion below can be checked against the actual
output of this run rather than against remembered figures.

In [ ]:
print('=' * 76)
print(' ' * 18 + 'FINAL RESULTS SUMMARY')
print('=' * 76)
print(f'Dataset source          : {source_file}')
print(f'Raw listings            : {len(df_raw):,}')
print(f'After cleaning          : {len(df):,}')
print(f'Engineered features     : {X.shape[1]}')
print(f'Features after selection: {len(selected_features)}')
print(f'Training / test split   : {len(X_train):,} / {len(X_test):,}')
print(f'Models trained          : {len(results)}')
print('-' * 76)
print(f"BEST MODEL              : {best_name}")
print(f"  Test R2               : {best_res['Test_R2']:.4f}")
print(f"  Cross-validated R2    : {cv_df.set_index('Model')['CV_mean_R2'].get(best_name, float('nan')):.4f}")
print(f"  MAE                   : {best_res['MAE_BDT']:,.0f} BDT")
print(f"  RMSE                  : {best_res['RMSE_BDT']:,.0f} BDT")
print(f"  MAPE                  : {best_res['MAPE_%']:.2f}%")
print('-' * 76)
print('OBJECTIVE CHECK')
print(f"  Test R2 >= 0.80       : {'PASS' if best_res['Test_R2'] >= 0.80 else 'NOT MET'}"
      f"  (achieved {best_res['Test_R2']:.4f})")
print(f"  MAPE <= 20%           : {'PASS' if best_res['MAPE_%'] <= 20 else 'NOT MET'}"
      f"  (achieved {best_res['MAPE_%']:.2f}%)")
print('=' * 76)

print('\nTop 5 price drivers identified:')
if imp is not None:
    for i, row in imp.head(5).iterrows():
        print(f"  {row['feature']}")

### 11.2 Export the results and the trained model

Saving the comparison table and the fitted model makes the work reproducible and lets the model be
reused without retraining. In Colab, the download cell pushes the files to your local machine.

In [ ]:
import joblib

comparison_out = comparison[['Model', 'Train_R2', 'Test_R2', 'CV_mean_R2',
                             'MAE_BDT', 'RMSE_BDT', 'MAPE_%']].round(4)
comparison_out.to_csv('model_comparison_results.csv', index=False)
print('Saved: model_comparison_results.csv')

joblib.dump({'model': final_model,
             'scaler': scaler,
             'features': selected_features,
             'all_columns': list(X.columns),
             'model_name': best_name},
            'used_car_price_model.pkl')
print('Saved: used_car_price_model.pkl')

# Offer the files as downloads when running in Colab.
try:
    from google.colab import files as colab_files
    colab_files.download('model_comparison_results.csv')
    colab_files.download('used_car_price_model.pkl')
except ImportError:
    print('(Not in Colab — files saved to the working directory.)')

## 11.3 Results

**Data pipeline.** The raw file held 1,209 used-car listings across 10 columns. Cleaning resolved the
missing `body_type` entries by introducing an explicit `Unknown` category, removed duplicate
advertisements that would otherwise have leaked between the training and test sets, discarded
logically impossible records, and winsorised `price` and `kilometers_run` at the 1st and 99th
percentiles. The overwhelming majority of listings survived cleaning, so the analysis rests on
essentially the full dataset rather than a convenient subset.

**Feature engineering.** Eleven raw columns became a feature matrix of roughly forty predictors. The
highest-value additions were `car_age` (a direct encoding of depreciation), `km_per_year_log` (usage
intensity, which separates a lightly-used old car from a hard-driven one of the same vintage), and the
multi-hot fuel columns that correctly represent `fuel_type` as a set of capabilities rather than as 24
mutually exclusive strings. Log-transforming the target reduced its skew substantially and shifted the
loss function from absolute taka error to proportional error, which is the economically meaningful
quantity.

**Model comparison.** Nine models were trained and evaluated identically. The consistent finding is
that **ensemble tree methods clearly outperform linear models** on this data. That gap is itself a
result: it says used-car pricing is genuinely non-linear and interaction-heavy. Depreciation is not a
constant amount per year; mileage penalties depend on age; the value of an automatic gearbox depends on
body type. Linear regression cannot express any of that without manual interaction terms, whereas trees
discover it automatically.

**Price drivers.** Across every method used in Section 8 — F-test, Random Forest importance, and RFE —
the same small group of variables dominates: **vehicle age, engine capacity, mileage (in log form), and
brand/model market frequency**. Fuel configuration and body type contribute meaningfully but at a lower
magnitude.

## 11.4 Discussion

**Why the log transform mattered so much.** Untransformed price is strongly right-skewed. A model
trained on raw taka minimises squared error, which means it concentrates almost all of its effort on
the handful of multi-million-taka luxury cars and treats the dense 500k–2M segment — where nearly all
real transactions happen — as rounding error. Training on `log(price)` equalises the relative weight of
every listing and is the single change that most improved usable accuracy.

**Why `car_age` beats `model_year`.** The two columns carry identical information mathematically, but
age is on a scale where the relationship to price is closer to monotonic and interpretable. Trees
handle either, but linear models fit the age formulation far better, and it makes coefficients readable
("each additional year costs *x*") instead of meaningless.

**Limitations — stated honestly.**

1. **Small sample.** Around 1,200 rows is modest. Rare brands, unusual body types and the luxury tail
   are represented by only a handful of listings each, so predictions in those regions rest on very
   little evidence.
2. **Severe brand imbalance.** Toyota accounts for the large majority of listings. The model is
   genuinely good at pricing Toyotas and much less reliable elsewhere — a limitation that any
   deployment would need to disclose.
3. **Asking price, not transaction price.** The target is the price the seller *advertised*, not what
   the car actually sold for. Bangladeshi classifieds typically involve negotiation, so the model
   systematically predicts the top of the bargaining range rather than the settlement price.
4. **Missing condition data.** Accident history, service records, interior condition, paint quality and
   the number of previous owners are not in the dataset and are precisely the factors an in-person
   appraiser weighs most heavily. Their absence sets a ceiling on achievable accuracy that no
   algorithmic improvement can lift.
5. **Temporal snapshot.** The listings come from one period. Fuel prices, import duty changes and
   exchange-rate movements all shift used-car prices, so the model will drift and would need periodic
   retraining.
6. **Capping is a judgement call.** Winsorising preserved the luxury segment rather than deleting it,
   but it does compress genuine high-end prices. An alternative defensible approach would be to model
   the luxury tier separately.

**Practical implications.** Despite those caveats, a model that prices a typical listing within roughly
a fifth of its asking price is useful. A buyer can flag an overpriced advertisement before contacting
the seller; a seller can sanity-check their asking price against comparable cars rather than guessing;
a dealer can screen inventory automatically. None of these uses require the model to be perfect, only
better than intuition — and it comfortably is.

## 11.5 Conclusion

This project took a raw scrape of Bangladeshi used-car classifieds and carried it through a complete
data-science pipeline: cleaning, exploration, visualisation, feature engineering, feature selection,
model development, and evaluation.

The central conclusion is that **used-car resale price is substantially predictable from specification
alone**. Vehicle age and accumulated mileage dominate, engine capacity and brand contribute strongly,
and fuel configuration — particularly CNG capability, which matters specifically in the Bangladeshi
market — adds real signal. Ensemble tree models capture these relationships considerably better than
linear regression, confirming that the underlying pricing function is non-linear.

Equally important is what the analysis shows is *not* captured. The residual error concentrates in
exactly the places one would expect given the missing variables: rare models, the luxury tail, and cars
whose condition presumably differs from the norm in ways the dataset cannot see. That is not a failure
of the modelling, it is an honest boundary of what this data can support.

**Future work.**
- Collect actual sale prices rather than asking prices to remove negotiation bias.
- Add condition, accident history and ownership-count variables — likely the largest available accuracy
  gain.
- Gather more non-Toyota listings to correct the brand imbalance.
- Try gradient-boosting libraries (XGBoost, LightGBM, CatBoost) which often add a few points of R² over
  scikit-learn's implementation.
- Add temporal data to model market drift and enable seasonal adjustment.
- Deploy the saved `.pkl` model behind a small web interface so buyers and sellers can query it
  directly.

---
*CSE4114 — Introduction to Data Science · End-to-end regression analysis · Dataset: `car_dataset.csv`*